# NB0 · 实验室导览：把课堂装进浏览器

欢迎来到 **RL 可视化课堂的笔记本实验室**。这里是 NB0——一本导览本，带你完成三件事：

1. **环境自检**：确认你正跑在浏览器内的 Python（Pyodide）里；
2. **复刻 4×4 GridWorld**：用 numpy 瘦身重写书配代码的环境类（只保留 `reset / step / _get_next_state_and_reward` 三个核心方法）；
3. **与站内作业世界对齐**：用 assert 逐条验证转移与奖励，和主站 L1「代码精讲」的 4×4 作业世界一模一样。

跑通全部格子，你就毕业了——NB1–NB6 的算法实验都在这个环境上展开。

## 怎么用这本笔记本

- 选中格子按 **Shift + Enter** 运行并跳到下一格；菜单 **Run → Run All Cells** 一键全跑；
- 带宽提示：**首次启动需下载 ~15MB 的 Python 内核（Pyodide）**，之后浏览器缓存，二次进入几乎秒开；
- 已知限制：**浏览器刷新会清空 kernel 状态**（变量、环境都要重跑）；笔记本文件的改动默认不落盘，想保留请用菜单下载备份；
- 全部格子只用 numpy——不加载 matplotlib，省下 8–10MB 的下载量。

## 0 · 环境自检

先看清楚自己脚下的地面：Python 版本、实现与平台。

In [ ]:
import sys
import platform

print("Python 版本 :", sys.version.split()[0])
print("实现        :", sys.implementation.name)
print("平台        :", platform.platform())
print("sys.platform:", sys.platform)

**在不在浏览器内核里？** 浏览器内核（Pyodide）里可以 `import pyodide`，本地 Python 则不行——这是最可靠的判据。另一个口径是 `sys.platform == "emscripten"`（WebAssembly 平台）。两个口径必须互相印证，我们在本地用 nbconvert 校验这本笔记本时走的是本地 Python 分支，在浏览器里打开时走 Pyodide 分支。

In [ ]:
IN_BROWSER = False
try:
    import pyodide  # noqa: F401  只有浏览器内核里能导入
    IN_BROWSER = True
except ImportError:
    pass

# 一致性断言：能否 import pyodide ⟺ 是否跑在 emscripten(wasm) 平台
assert IN_BROWSER == (sys.platform == "emscripten")
print("运行环境 :", "浏览器（Pyodide / WebAssembly）" if IN_BROWSER else "本地 Python（nbconvert 校验模式）")

In [ ]:
import numpy as np

print("numpy 版本 :", np.__version__)
major = int(np.__version__.split(".")[0])
assert major in (1, 2), "numpy 主版本异常"
print("numpy 就绪（纯 numpy，无 matplotlib）")

## 1 · 热身：折扣回报与 γ

第一讲反复出现的一行公式——回报是奖励的折扣和：

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots$$

先用 numpy 向量化实现它，后面采样轨迹时直接复用。

In [ ]:
GAMMA = 0.9  # 主站 A4 作业世界的折扣因子

def discounted_return(rewards, gamma=GAMMA):
    """G_t = r1 + gamma*r2 + gamma^2*r3 + ...（numpy 点积实现）"""
    r = np.asarray(rewards, dtype=float)
    weights = gamma ** np.arange(len(r))
    return float(r @ weights)

demo = [0, 0, 0, 1]   # 走三步才拿到 +1
print("G0 =", discounted_return(demo))
assert abs(discounted_return(demo) - GAMMA ** 3) < 1e-12
assert abs(discounted_return([1, 1, 1]) - (1 + GAMMA + GAMMA ** 2)) < 1e-12
print("三步之外的 +1 只值", round(GAMMA ** 3, 4), "——γ 让智能体偏爱早到的奖励")

## 2 · 4×4 GridWorld：作业世界规格

这就是主站 L1「代码精讲」反复打磨的**作业世界**（站内 `assets/js/data.js` 顶部的 `A4` 常量），规格如下：

| 项目 | 值 |
|---|---|
| 网格 | **4×4**，状态编号 s1–s16 |
| 起点 | **s1**（左上角） |
| 禁区 | **s8、s10** |
| 目标 | **s12** |
| 奖励 | 边界 **−1** / 禁区 **−1** / 目标 **+1** / 其他 **0** |
| 折扣 | **γ = 0.9** |

编号规则：**s_i ↔ (x, y) = ((i−1) % 4, (i−1) // 4)**，y 向下增长（第 1 行是 y=0）。于是 s8=(3,1)、s10=(1,2)、s12=(3,2)。

动作空间 **5 个**，采用作业代码的列序（**下、右、上、左、原**）——注意它与书里 a1–a5（上右下左原）**列序不同**，作业策略矩阵以代码为准。

In [ ]:
SIZE = 4
NUM_STATES = SIZE * SIZE          # s1 .. s16
START, TARGET = 1, 12
FORBIDDEN = {8, 10}
REWARDS = {"boundary": -1.0, "forbidden": -1.0, "target": 1.0, "other": 0.0}

# 与主站 data.js 的 A4 作业配置逐项对齐
assert (SIZE, NUM_STATES, START, TARGET) == (4, 16, 1, 12)
assert FORBIDDEN == {8, 10}
assert REWARDS == {"boundary": -1.0, "forbidden": -1.0, "target": 1.0, "other": 0.0}
assert GAMMA == 0.9
print("规格自检通过：4×4 / 起点 s1 / 禁区 s8,s10 / 目标 s12 / γ=0.9")

In [ ]:
ACTION_SPACE = [(0, 1), (1, 0), (0, -1), (-1, 0), (0, 0)]  # 下 右 上 左 原
DOWN, RIGHT, UP, LEFT, STAY = ACTION_SPACE

assert len(ACTION_SPACE) == 5            # 动作空间 5 个
assert len(set(ACTION_SPACE)) == 5
print("动作空间（作业代码列序）:", ACTION_SPACE)

In [ ]:
def s2xy(s):
    """状态编号 → (x, y)，y 向下增长：s1=(0,0) s8=(3,1) s10=(1,2) s12=(3,2)"""
    i = int(s) - 1
    return i % SIZE, i // SIZE

def xy2s(x, y):
    """(x, y) → 状态编号"""
    return int(y) * SIZE + int(x) + 1

assert s2xy(1) == (0, 0) and s2xy(8) == (3, 1) and s2xy(10) == (1, 2) and s2xy(12) == (3, 2)
assert all(xy2s(*s2xy(s)) == s for s in range(1, 17))
print("坐标换算就绪：s8=(3,1)  s10=(1,2)  s12=(3,2)")

## 3 · 环境类：书配代码的 numpy 瘦身版

底本是《Mathematical Foundation of RL》书配仓库 `Code for grid world/python_version/src/grid_world.py`（西湖大学智能无人系统实验室）。瘦身原则：**自包含单文件、只依赖 numpy**——

- 砍掉 `render / add_policy / add_state_values`（matplotlib 可视化主站已经有了）；
- 砍掉 `examples/arguments.py` 的命令行配置，世界参数直接内联成上面的常量；
- **保留** `reset / step / _get_next_state_and_reward` 三方法，转移与奖励全用 numpy 实现。

语义要点（与书配代码一致，也是主站 L1 强调的「作业代码规则」）：分支优先级 **出界 > 目标 > 禁区 > 普通**，if/elif 短路保证每个动作只命中第一个匹配分支；**禁区是弹回的**（原地不动、挨罚 −1），与课上「可进入但扣分」的书规则不同。

In [ ]:
class GridWorld:
    """4×4 网格世界：书配 grid_world.py 的 numpy 瘦身复刻（自包含单文件版）。

    语义（作业代码规则）：
      出界    → 原地不动，reward = boundary  = -1
      进目标  → 走进目标，reward = target   = +1，done=True
      撞禁区  → 原地弹回，reward = forbidden = -1
      普通/原 → 正常移动，reward = other    =  0
    """

    def __init__(self, size=SIZE, start=START, target=TARGET, forbidden=FORBIDDEN):
        self.size = size
        self.num_states = size * size
        self.start_state, self.target_state = start, target
        self.forbidden_states = set(forbidden)
        self.action_space = ACTION_SPACE
        self.agent_state = start

    def reset(self):
        self.agent_state = self.start_state
        return self.agent_state

    def _get_next_state_and_reward(self, state, action):
        x, y = s2xy(state)
        nxt = np.array([x, y]) + np.array(action)          # numpy 实现转移
        if not (0 <= nxt[0] < self.size and 0 <= nxt[1] < self.size):
            next_state, reward = state, REWARDS["boundary"]      # 1) 出界：原地
        elif xy2s(nxt[0], nxt[1]) == self.target_state:
            next_state, reward = xy2s(nxt[0], nxt[1]), REWARDS["target"]   # 2) 目标
        elif xy2s(nxt[0], nxt[1]) in self.forbidden_states:
            next_state, reward = state, REWARDS["forbidden"]     # 3) 禁区：弹回
        else:
            next_state, reward = xy2s(nxt[0], nxt[1]), REWARDS["other"]    # 4) 普通
        return next_state, reward

    def step(self, action):
        assert any(tuple(action) == a for a in self.action_space), "非法动作"
        next_state, reward = self._get_next_state_and_reward(self.agent_state, action)
        done = next_state == self.target_state
        self.agent_state = next_state
        return next_state, reward, done, {}

env = GridWorld()
s0 = env.reset()
assert s0 == START == 1
assert env.num_states == 16 and len(env.action_space) == 5
print("环境就绪：reset() → s", s0)

## 4 · 对齐验证：与站内作业世界逐条 assert

下面三条断言对应主站 L1 代码精讲里的锚点案例——**全绿说明这个环境与站内作业世界严丝合缝**：

1. s1 向上（出界）：reward = −1，留在 s1；
2. s11 右移进 s12（目标）：reward = +1 且 done；
3. s7 右移撞 s8（禁区）：reward = −1（作业代码规则：弹回原地）。

In [ ]:
env.reset()
ns, r, done, _ = env.step(UP)          # s1 向上：出界
print(f"s1  --up-->    s{ns}   reward={r:+.0f}  done={done}")
assert ns == 1 and r == -1.0 and done is False, "出界：−1 且留在 s1"

In [ ]:
env.agent_state = 11                   # 摆到 s11（s12 的左邻）
ns, r, done, _ = env.step(RIGHT)       # s11 右移进 s12
print(f"s11 --right--> s{ns}   reward={r:+.0f}  done={done}")
assert ns == 12 and r == 1.0 and done is True, "进目标：+1 且 done"

In [ ]:
env.agent_state = 7                    # 摆到 s7（s8 的左邻）
ns, r, done, _ = env.step(RIGHT)       # s7 右移撞 s8 禁区
print(f"s7  --right--> s{ns}   reward={r:+.0f}  done={done}")
assert r == -1.0 and ns == 7, "撞禁区：−1 且弹回原地（作业代码规则）"

单点对齐还不够——把 **16 状态 × 5 动作 = 80 条转移**全部扫一遍，验证不变式：下一状态永远合法、奖励只出自 {−1, 0, +1}、边界与禁区的行为处处一致。

In [ ]:
transitions = {}
for s in range(1, NUM_STATES + 1):
    for a, name in zip(ACTION_SPACE, ["下", "右", "上", "左", "原"]):
        ns, r = env._get_next_state_and_reward(s, a)
        transitions[(s, name)] = (ns, r)

assert all(1 <= ns <= 16 for ns, _ in transitions.values())       # 下一状态合法
assert {r for _, r in transitions.values()} == {-1.0, 0.0, 1.0}   # 奖励只有三档
assert transitions[(16, "右")] == (16, -1.0)     # 出界：右边界右移
assert transitions[(13, "下")] == (13, -1.0)     # 出界：下边界下移
assert transitions[(1, "左")] == (1, -1.0)       # 出界：左上角左移
assert transitions[(4, "下")] == (4, -1.0)       # 撞 s8 弹回
assert transitions[(11, "左")] == (11, -1.0)     # 撞 s10 弹回
assert transitions[(11, "右")] == (12, 1.0)      # 进 s12
assert transitions[(16, "上")] == (12, 1.0)      # 从 s16 向上进 s12
assert transitions[(12, "原")] == (12, 1.0)      # 停在目标：+1
print("80 条转移全部扫描，不变式抽查 9 条全部通过")

## 5 · 张量化：转移张量 T 与奖励矩阵 R

站内只有 A4 这种「规格常量」，没有现成的 T/R——现在由环境现场构造：

- **T[s, a, s′]**：确定性转移的 one-hot（每行恰有一个 1）；
- **R[s, a]**：即时奖励。

有了这两个张量，Bellman 方程就能写成矩阵运算——这正是 NB1（手算 Bellman + 迭代 vs 闭式解）的地基。

In [ ]:
T = np.zeros((NUM_STATES, 5, NUM_STATES))   # T[s-1, a, s'-1]
R = np.zeros((NUM_STATES, 5))
for s in range(1, NUM_STATES + 1):
    for a in range(5):
        ns, r = env._get_next_state_and_reward(s, ACTION_SPACE[a])
        T[s - 1, a, ns - 1] = 1.0
        R[s - 1, a] = r

assert T.shape == (16, 5, 16) and R.shape == (16, 5)
assert np.allclose(T.sum(axis=-1), 1.0)          # one-hot：转移概率合法
assert T[0, 2, 0] == 1.0 and R[0, 2] == -1.0     # s1 上 → s1，−1
assert T[10, 1, 11] == 1.0 and R[10, 1] == 1.0   # s11 右 → s12，+1
assert T[6, 1, 6] == 1.0 and R[6, 1] == -1.0     # s7 右 → 弹回 s7，−1
print("T:", T.shape, " R:", R.shape, "——动力学张量化完成")

## 6 · 预演 Bellman：均匀随机策略的策略评估

第二讲的主方程——Bellman 期望方程的矩阵形式：

$$v_\pi = r_\pi + \gamma P_\pi v_\pi$$

取均匀随机策略 π(a|s) = 0.2，从零向量开始迭代；同时用闭式解 $v = (I - \gamma P_\pi)^{-1} r_\pi$ 验算。两条路殊途同归，是 NB1 的先尝版。

In [ ]:
PI_uniform = np.full((NUM_STATES, 5), 0.2)             # π(a|s) = 1/5
P_pi = np.einsum("sa,sat->st", PI_uniform, T)             # 16×16 转移矩阵
r_pi = np.einsum("sa,sa->s", PI_uniform, R)               # 期望即时奖励

v = np.zeros(NUM_STATES)
for k in range(5000):
    v_new = r_pi + GAMMA * (P_pi @ v)
    if np.abs(v_new - v).max() < 1e-13:
        break
    v = v_new
v_iter = v_new
v_closed = np.linalg.solve(np.eye(NUM_STATES) - GAMMA * P_pi, r_pi)

assert np.abs(v_iter - v_closed).max() < 1e-9, "迭代解应收敛到闭式解"
print(f"迭代 {k + 1} 轮收敛；迭代 vs 闭式解最大差 = {np.abs(v_iter - v_closed).max():.2e}")
print("v(s1) =", round(float(v_iter[0]), 6), "   v(s12) =", round(float(v_iter[11]), 6))

In [ ]:
print("均匀随机策略下的状态价值 v(s)：")
print(np.round(v_iter.reshape(SIZE, SIZE), 3))
print()
print("看两点：s1 附近价值为负（随机策略常撞墙）；s12 的价值也只有")
print(round(float(v_iter[11]), 3), "——站在目标上，均匀策略 4/5 的动作都在挨罚。")

## 7 · 采一批轨迹：随机策略现场走一遭

最后把动力学「跑起来」：用固定种子采样 300 条轨迹，比较**平均折扣回报**与上一步算出的 **v(s1)**。一条样本 ≠ 期望，但多条样本的平均会靠近期望——这正是第五讲「蒙特卡洛方法」与「方差是第一公民」的主题，NB3 会在这里深挖。

In [ ]:
rng = np.random.default_rng(20260913)
MAX_STEPS = 400
returns, hits = [], 0
for _ in range(300):
    env.reset()
    rewards = []
    for _ in range(MAX_STEPS):
        a = ACTION_SPACE[int(rng.integers(5))]     # 均匀随机策略
        _, r, done, _ = env.step(a)
        rewards.append(r)
        if done:
            hits += 1
            break
    returns.append(discounted_return(rewards))

mean_G0 = float(np.mean(returns))
print(f"300 条轨迹：{hits} 条到达目标；平均 G0 = {mean_G0:.4f}")
print(f"对照理论值  v(s1) = {float(v_iter[0]):.4f}，偏差 {abs(mean_G0 - float(v_iter[0])):.4f}")
assert hits > 0, "固定种子下随机策略应能撞到目标"
assert abs(mean_G0 - float(v_iter[0])) < 0.2, "300 条样本的均值应落在期望附近"

## 8 · 回顾：你刚刚做了什么

- **环境自检**：确认了运行时（本地校验 or 浏览器 Pyodide）、numpy 就绪；
- **复刻环境**：numpy 单文件重写了书配 GridWorld，保留 reset / step / _get_next_state_and_reward；
- **对齐验证**：规格常量 + 三条锚点断言 + 80 条转移不变式，与站内 A4 作业世界严丝合缝；
- **张量化**：构造 T(16×5×16) 与 R(16×5)；
- **预演 Bellman**：迭代解 == 闭式解；采样均值 → v(s1)。

工具链全部就位。接下来，算法正式登场。

## 🎓 跑通即毕业

到这里全部格子跑绿——说明浏览器里的 Python、numpy、环境复刻、与主站的对齐全部成立。**你已毕业，实验室正式对你开放。**

后续六本实验笔记本（陆续上线）：

| 本 | 对应讲 | 主线 |
|---|---|---|
| NB1 | L2/L3 | 手算 Bellman 一轮 → 编程迭代验证；闭式解 vs 迭代 |
| NB2 | L4 | 值迭代 / 策略迭代实现；γ ∈ {0.5, 0.9, 0.99} 扫描 |
| NB3 | L5/L6 | MC 多 seed 方差实验；Robbins-Monro 步长对比 |
| NB4 | L7 | Q-learning（seed 可调）+ α/ε 敏感性 + 10-seed 误差带 |
| NB5 | L8 | TD-Linear 线性值近似 + 特征设计消融 |
| NB6 | L9/L10 | REINFORCE + baseline 消融（DQN 真网络给 Colab 链接） |

提醒：**刷新浏览器会清空 kernel 状态**——离开前想保留改动，请下载笔记本；回来后 Run All 重跑一遍即可。